# 01 · Data prep

Loads the StockTwits crypto dataset, checks what its labels mean, cleans it, and splits it into a training set, a validation set and a held-out pool for the test set.

- **Input:** `st-data-full.xlsx` from [`ElKulako/stocktwits-crypto`](https://huggingface.co/datasets/ElKulako/stocktwits-crypto) on Hugging Face (downloaded and cached automatically)
- **Outputs:**
  - `data/processed/train.csv`: 30,000 posts (10,000 per label), for fine-tuning
  - `data/processed/val.csv`: 3,000 posts (1,000 per label), for checking training progress
  - `data/processed/test_pool.csv`: 15,000 posts (5,000 per label); notebook 02 draws the hand-labeled test set from here
  - `data/interim.nosync/clean_posts.csv`: the full cleaned dataset (not committed); notebook 02 uses it to measure how common slang is

## Setup

In [1]:
import re
import sys
import warnings
from pathlib import Path

import pandas as pd

# Hide a harmless notice about notebook progress bars, printed when huggingface_hub is imported
warnings.filterwarnings("ignore", message="IProgress not found")
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import logging as hf_logging

# Let the notebook import our shared code from src/
ROOT = Path("..").resolve()
sys.path.append(str(ROOT))
from src.labels import DATASET_LABELS, LABELS

# Hide a Hugging Face notice suggesting a login token; this dataset is public
hf_logging.set_verbosity_error()

SEED = 42
TEST_POOL_PER_CLASS = 5_000
TRAIN_PER_CLASS = 10_000
VAL_PER_CLASS = 1_000
MIN_WORDS = 4

## 1. Load the dataset

The dataset is a single Excel file with two sheets. `hf_hub_download` stores it in the Hugging Face cache (outside this project folder), so it only downloads once.

Each post gets a `post_id`: its row number in the file (sheet 1, then sheet 2). Every later file refers to posts by this ID.

Labels are stored as 0, 1, 2. CryptoBERT's model card uses the same coding (0 = Bearish, 1 = Neutral, 2 = Bullish), so we convert them to names right away.

In [2]:
path = hf_hub_download("ElKulako/stocktwits-crypto", "st-data-full.xlsx", repo_type="dataset")
sheets = pd.read_excel(path, sheet_name=None)  # None = read every sheet

raw = pd.concat(sheets.values(), ignore_index=True)
raw["post_id"] = raw.index
raw["label"] = raw["label"].map(DATASET_LABELS)

print(f"{len(raw):,} posts from sheets {list(sheets)}")

1,331,697 posts from sheets ['stocktwits_1', 'stocktwits_2']


## 2. Clean

Each step prints how many posts it removes.

1. **Empty rows.** A few cells are empty or hold just the number 0.
2. **Exact duplicates.** The dataset card says duplicates were removed, but copies that differ only by leading spaces are still there. When copies disagree on the label, the first copy is kept.
3. **Near duplicates.** Posts that are the same apart from numbers, punctuation or emoji. Bots often repost one message with only the price changed (*"going to 32k by monday"*, *"going to 10k by monday"*). Removing them also means a near-copy can't end up in both the training set and the test pool.
4. **Very short posts** (under 4 words). The dataset author already removed these, so this step is just a check.

In [3]:
def dedup_key(text):
    """Letters-only version of a post, so reposts that change only numbers or punctuation match."""
    letters_only = " ".join(re.sub(r"[^a-z ]", " ", text).split())
    # Posts that are mostly emoji or numbers keep their full text, so they aren't all lumped together
    if len(letters_only.split()) < 3:
        return text
    return letters_only


dedup_key("going to 32k by monday 🚀"), dedup_key("💎 ✋ ✋ 🚀🌛")

('going to k by monday', '💎 ✋ ✋ 🚀🌛')

In [4]:
removed = {}

df = raw[raw["text"].map(lambda t: isinstance(t, str))].copy()
removed["empty rows"] = len(raw) - len(df)
df["text"] = df["text"].str.strip()

before = len(df)
df = df.drop_duplicates("text")  # keeps the first copy
removed["exact duplicates"] = before - len(df)

before = len(df)
df["dedup_key"] = df["text"].map(dedup_key)
df = df.drop_duplicates("dedup_key")
removed["near duplicates"] = before - len(df)

before = len(df)
df = df[df["text"].str.split().str.len() >= MIN_WORDS]
removed[f"under {MIN_WORDS} words"] = before - len(df)

clean = df[["post_id", "text", "label"]].reset_index(drop=True)

print(pd.Series(removed, name="posts removed").to_string())
print(f"\n{len(raw):,} posts -> {len(clean):,} after cleaning")

empty rows            151
exact duplicates    39412
near duplicates     37223
under 4 words           0

1,331,697 posts -> 1,254,911 after cleaning


## 3. What do the labels mean?

On StockTwits, an author can tag a post **Bullish** or **Bearish**, or leave it untagged. There is no "neutral" button. The dataset author [confirmed on Hugging Face](https://huggingface.co/datasets/ElKulako/stocktwits-crypto/discussions/3) how the labels were made:

> "The posts were labelled by their authors as either Bullish or bearish. If no label was given, we assumed the post to be neutral."

So in this dataset, **neutral means "untagged"**, not "has no opinion". The classes are also very uneven:

In [5]:
counts = clean["label"].value_counts()
pd.DataFrame({"posts": counts, "share": (counts / len(clean)).round(3)})

,posts,share
label,,
bullish,636245,0.507
neutral,499844,0.398
bearish,118822,0.095


A few random posts per label show what this looks like in practice:

In [6]:
for label in LABELS:
    print(f"--- {label} ---")
    for text in clean.loc[clean["label"] == label, "text"].sample(5, random_state=SEED):
        print("  ", text[:110])

--- bullish ---
   looks to me like we found our path to
   low volume big moves i'll hold 💸
   like damn ya just gone make a playa average up lol
   nft rising up (plin changed ticker into btog)
   why yall even care so much about gme ? 🙄
--- bearish ---
   annnnnnnnnnnnd 10% drop less than 3 hours 🤢🤮
   whys it heading downward, now?
   bull trap... won't be enough exit liquidity to let everyone out
   last minute candle was 5 btc. this is a trap
   black swan inc... you are picking up pennies in front of a steam roller.
--- neutral ---
   oh no panic and sell everything!
   russia is making moves bitcoin is a safehaven for countries
   come on ffs eth doesnt trend up like this everyday... lets follow that bitchh...
   if the market is forward looking shouldn’t it start to recover if the market will be significantly higher by n
   - you little terds are literally on stand-by waiting for red candles. weak.


**What this means for the project.**

- The tags are **noisy**. Untagged posts often still carry an opinion, and some tagged posts don't match their tag (authors may tag out of habit or sarcasm).
- The fine-tuned model learns from these tags, so its "neutral" class really learns "posts people didn't tag". CryptoBERT was trained on the same tags, which keeps that comparison fair.
- This is why the **test set is labeled by hand** (notebook 02). The hand labels measure real sentiment, and comparing them with the author tags shows how noisy the tags are.

## 4. Split

The held-out pool is set aside **first**, so no training or validation post can end up in the test set. Each split is **balanced**, with the same number of posts per label. The raw data is about half bullish and under 10% bearish, so a model trained on it as-is could score well just by leaning toward "bullish".

The training set is a 30k sample, not the full 1.2M posts. That's enough to fine-tune with LoRA and keeps training time on a free Colab GPU manageable.

In [7]:
def take_balanced(posts, per_class):
    """Randomly pick `per_class` posts from each label, then shuffle."""
    picked = posts.groupby("label").sample(n=per_class, random_state=SEED)
    # Shuffle so the file isn't sorted by label
    return picked.sample(frac=1, random_state=SEED).reset_index(drop=True)


test_pool = take_balanced(clean, TEST_POOL_PER_CLASS)
rest = clean[~clean["post_id"].isin(test_pool["post_id"])]

train = take_balanced(rest, TRAIN_PER_CLASS)
rest = rest[~rest["post_id"].isin(train["post_id"])]

val = take_balanced(rest, VAL_PER_CLASS)

splits = {"train": train, "val": val, "test_pool": test_pool}
pd.DataFrame({name: split["label"].value_counts() for name, split in splits.items()}).loc[LABELS]

,train,val,test_pool
label,,,
bullish,10000,1000,5000
bearish,10000,1000,5000
neutral,10000,1000,5000


Check that no post, or near-copy of a post, appears in two splits:

In [8]:
for a, b in [("train", "val"), ("train", "test_pool"), ("val", "test_pool")]:
    shared_ids = set(splits[a]["post_id"]) & set(splits[b]["post_id"])
    shared_keys = set(splits[a]["text"].map(dedup_key)) & set(splits[b]["text"].map(dedup_key))
    assert not shared_ids and not shared_keys, f"{a} and {b} overlap"
print("No overlap between splits.")

No overlap between splits.


## 5. Save

In [9]:
processed_dir = ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
for name, split in splits.items():
    split.to_csv(processed_dir / f"{name}.csv", index=False)
    print(f"data/processed/{name}.csv: {len(split):,} posts")

# The full cleaned dataset is large, so it goes in a .nosync folder (skipped by iCloud and git)
interim_dir = ROOT / "data" / "interim.nosync"
interim_dir.mkdir(parents=True, exist_ok=True)
clean.to_csv(interim_dir / "clean_posts.csv", index=False)
print(f"data/interim.nosync/clean_posts.csv: {len(clean):,} posts")

data/processed/train.csv: 30,000 posts
data/processed/val.csv: 3,000 posts
data/processed/test_pool.csv: 15,000 posts


data/interim.nosync/clean_posts.csv: 1,254,911 posts


## Summary

- Loaded **1,331,697** posts and kept **1,254,911** after removing empty rows, about 39k exact duplicates and 37k near duplicates.
- The labels are the authors' own tags, and **neutral means untagged**. The cleaned data is 51% bullish, 40% neutral and under 10% bearish.
- Built balanced splits: 30k training, 3k validation and a 15k held-out pool, with no post (or near-copy) shared between them.